# Day 23 — FastAPI First Steps: Routes, Path and Query Parameters, Pydantic Models, Validation Errors & /docs (16 Nov)

*Rebuilt from the master Google Doc session (lesson text and every code example). Code cells are not executed: several need Postgres, a running server or files from the original starter zip.*

## Today at a glance

- Goal: turn yesterday's REST design into a running API. By tonight you can write FastAPI routes, read path and query parameters, define request and response models with Pydantic, return the right status codes, and explain every field of a FastAPI validation error.

- Deliverable: day23/content_api.py (the entries part of the Acme content API, stored in memory) passing the 26 given tests in day23/test_content_api.py, plus day23/smoke.sh (9 curl checks against the running server). DSA in day23/dsa.py.

- Where this sits: Week 4, Phase 1 (Foundation), day 2 of 7. Tomorrow the in-memory dict is replaced by your Week 3 Postgres schema, Day 25 adds a full test suite, Day 26 a webhook, and Day 27 packages it as the Phase 1 milestone. Every AI service you build from Week 5 (a RAG endpoint, an agent API, an MCP server's HTTP side) is a FastAPI app with exactly today's parts.

Time split (5h): the Week 4 split from the Day 21 plan.

| Block | Time | What you do |
|---|---|---|
| Theory | 1.5h | Five concepts below, typing every example; check-your-understanding predictions |
| Core task | 2h | content_api.py until the tests pass (90 min), smoke.sh and /docs (30 min) |
| Catch-up | 1h | Day 22 api_design.md if it is not pushed; otherwise Day 15 schema.sql and queries.sql |
| DSA | 0.5h | match_route and unique_slug in dsa.py |

Setup (10 min): make a fresh virtual environment inside day23/ and install FastAPI with its standard extras. Type the examples below into files in day23/scratch/ and run each with python3 <file>; only Concept 1 needs a running server.

```bash
# One-time setup for Day 23, inside day23/ (macOS / Linux shell).
python3 -m venv .venv            # a fresh venv for this API (Day 6)
source .venv/bin/activate        # Windows: .venv\Scripts\activate
# The quotes stop the shell from reading [ ] as a pattern.
# [standard] adds uvicorn (the server) and the "fastapi" command.
python3 -m pip install "fastapi[standard]" pytest
fastapi --version
# -> FastAPI CLI version: 0.0.34   (yours may be newer; that is fine)
python3 -c "import fastapi; print(fastapi.__version__)"
# -> 0.163.1   (any 0.11x or newer works for today)
```

## Recap (2 minutes, from memory)

- A valid read-only API key sends DELETE /v1/entries/blt003. Is the right answer 401 or 403, and what is the one-line rule that decides?

- An entry has a title and three tags. A client sends PUT with only {"title": "New"}, and another client sends PATCH with the same body. What should each leave on the server?

- A client retries a POST after a timeout. Where in the client code must the Idempotency-Key be created, and what goes wrong if it is created anywhere else?

## Concept 1: What FastAPI is, and your first app

### Why it matters for an FDE

FastAPI is the default Python web framework for AI services in 2026. When a customer asks for "an API in front of the model", a webhook receiver for their CMS, or an internal tool their other systems can call, you will very likely write it in FastAPI, and you will often read FastAPI code that the customer's team or another vendor wrote. It is fast to write, it validates input for you, and it produces interactive documentation that you can hand to a customer on the first day.

### How it works

Three layers work together. uvicorn is the server: it opens the port, reads raw HTTP bytes and passes each request to your app through a standard Python interface called ASGI (the async successor of WSGI; Day 13's async ideas are why it can handle many connections at once). FastAPI is the framework: it matches the method and path to one of your functions, converts and validates the inputs, calls the function, and turns the return value into a JSON response. Pydantic does the validation and conversion (Concept 3).

If you know Express, the shape is familiar: one app object, one handler per method and path. The difference is that FastAPI reads your type hints (Day 5) and uses them to decide what to validate and what to document. A route is a decorator (Day 11) on a plain function: @app.get("/health") registers the function below it for GET /health.

In [ ]:
# Your first FastAPI app: one file, two routes.
from fastapi import FastAPI  # the framework; like `import express` in Node

# One app object per service, like `const app = express()`.
app = FastAPI(title="Acme Content API", version="0.1.0")


# A decorator (Day 11) registers the function below as the handler
# for GET /health. Like app.get("/health", handler) in Express.
@app.get("/health")
def health() -> dict[str, str]:
    # Return a plain dict; FastAPI turns it into a JSON response.
    return {"status": "ok"}


@app.get("/")
def root() -> dict[str, str]:
    return {"message": "Acme Content API. See /docs"}

Run it with the fastapi command, which starts uvicorn for you. The examples below are real outputs from this app.

```bash
# Terminal 1: run the app with auto-reload (restarts on every save,
# like "npm run dev" with Vite). Save the code above as main.py first.
fastapi dev main.py
# -> Server started at http://127.0.0.1:8000
# -> Documentation at http://127.0.0.1:8000/docs

# Terminal 2: talk to it with Day 22's curl.
curl -s -i http://127.0.0.1:8000/health
# -> HTTP/1.1 200 OK
# -> content-type: application/json
# -> {"status":"ok"}
curl -s http://127.0.0.1:8000/nope
# -> {"detail":"Not Found"}            404, and you wrote no code for it
curl -s -i -X POST http://127.0.0.1:8000/health | grep -iE "^HTTP|^allow"
# -> HTTP/1.1 405 Method Not Allowed
# -> allow: GET                        the Allow header, also for free
# FDE TIP: in production you run "fastapi run main.py" (no reload, binds
# 0.0.0.0). "dev" is for your laptop only.
```

Open http://127.0.0.1:8000/docs in a browser. That page (Swagger UI) is generated from your code: every route, every parameter, every model. Its source is http://127.0.0.1:8000/openapi.json, an OpenAPI 3.1 document. That JSON file is the machine-readable version of the design you wrote yesterday, and tools can generate a TypeScript or Python client from it. /redoc shows the same spec in a reading-friendly layout.

### Real-world FDE example

On the first week of a deployment at a logistics customer, their integration team needs to start building against your service before it is finished. You deploy the skeleton with the real routes and models, and send them the /docs link. They try every endpoint in the browser, generate a typed client from /openapi.json, and raise two questions about field names on day two instead of week three. The documentation never drifts from the code because it is the code.

### Common mistakes

- Running python3 main.py and expecting a server. A FastAPI file only defines the app; fastapi dev main.py (or uvicorn main:app) runs it.

- Using fastapi dev in production. It reloads on every file change and listens only on localhost. Use fastapi run (Day 27 and Week 9 Docker).

- Two routes with the same method and path in one file. FastAPI does not warn you; the first one registered wins.

- Naming your file fastapi.py. Python then imports your file instead of the library (the Day 4 module search order).

## Concept 2: Path and query parameters

### Why it matters for an FDE

Most of what a client tells your API arrives in the URL: which entry (/v1/entries/blt001), which filter (?status=published), which page (?limit=20&cursor=blt040). In plain Python you would split strings and convert types by hand, and every endpoint would do it slightly differently. FastAPI does it from one line of type hints and rejects bad values with a 422 before your code runs, so your function only ever sees clean data.

### How it works

The rule is simple. A name in curly braces in the path, such as {entry_id}, is a path parameter: FastAPI passes that segment to the function argument with the same name. Every other simple argument (str, int, bool, a Literal) is a query parameter. An argument with a default is optional; without one, it is required. The type hint decides the conversion: "7" becomes 7 for int, "true" becomes True for bool.

Extra rules go in Annotated[type, Query(...)]: ge and le (greater or equal, less or equal) for numbers, min_length, max_length and pattern for strings. A list[str] query parameter collects repeated keys, which is how ?tag=sale&tag=diwali works. Literal["draft", "published"] allows only those strings, like a TypeScript union type.

In [ ]:
# Path and query parameters: FastAPI reads them from the type hints.
from typing import Annotated, Literal

from fastapi import FastAPI, Query
from fastapi.testclient import TestClient  # calls the app without a server

app = FastAPI()


# {entry_id} in the path becomes the function argument of the same name.
# The hint `int` makes FastAPI convert "7" -> 7, or reply 422 if it can't.
@app.get("/v1/items/{entry_id}")
def get_item(entry_id: int) -> dict:
    return {"entry_id": entry_id, "type": type(entry_id).__name__}


# Arguments NOT in the path are query parameters (?status=...&limit=...).
# A default value makes them optional. Literal limits the allowed values.
@app.get("/v1/items")
def list_items(
    status: Literal["draft", "published"] | None = None,
    # Annotated[type, rules]: type first, then extra validation rules.
    limit: Annotated[int, Query(ge=1, le=50)] = 3,
    tag: Annotated[list[str] | None, Query()] = None,  # ?tag=a&tag=b
) -> dict:
    return {"status": status, "limit": limit, "tag": tag}


client = TestClient(app)  # like supertest(app) in Node
print(client.get("/v1/items/7").json())
# -> {'entry_id': 7, 'type': 'int'}   "7" in the URL arrived as an int
print(client.get("/v1/items/seven").status_code)
# -> 422   not a whole number, so FastAPI rejects it before your code runs
print(client.get("/v1/items").json())
# -> {'status': None, 'limit': 3, 'tag': None}   all defaults
print(client.get("/v1/items?status=published&limit=10").json())
# -> {'status': 'published', 'limit': 10, 'tag': None}
print(client.get("/v1/items?tag=sale&tag=diwali").json()["tag"])
# -> ['sale', 'diwali']   repeated keys collect into a list
r = client.get("/v1/items?limit=500")
print(r.status_code, r.json()["detail"][0]["msg"])
# -> 422 Input should be less than or equal to 50

TestClient (used in every example from here on) sends requests to the app in-process, without a server or a port. It uses the same API as requests (Day 6), so client.get(url).json() reads the way you already know. Day 25 builds a full test suite with it.

### Real-world FDE example

A customer's storefront calls your search endpoint as /v1/products?limit=100000 during a sale, and an older service you inherited loads every row into memory and falls over. In your FastAPI version, limit: Annotated[int, Query(ge=1, le=50)] = 20 makes that request a 422 with a clear message, the documented maximum shows up in /docs, and nobody has to remember to write the check in each endpoint.

### Common mistakes

- Typing a path parameter as int when the ids are strings like blt001. Every request then gets a 422. Match the type to the real ids.

- Route order. Routes are checked top to bottom, so /v1/entries/{uid} declared before /v1/entries/latest swallows "latest" as a uid. Declare fixed paths first (the check below shows it).

- Making a filter required by forgetting = None. A list endpoint with a required status cannot list everything.

- Writing the limit check by hand inside the function, so it is missing from /docs and different in every endpoint.

## Concept 3: Pydantic models, the contract in code

### Why it matters for an FDE

Data that crosses a boundary (an HTTP body, a webhook payload, a CMS export, an LLM's JSON output in Week 6) must be checked before you trust it. Pydantic is how Python checks it: you describe the shape once as a class, and every input is validated and converted against it. FastAPI, the Anthropic and OpenAI SDKs, and most agent frameworks are built on Pydantic, so this is a skill you will use every day from now on, not just in web code.

### How it works

A model is a class that inherits from BaseModel, with one typed field per line. It looks like the Day 5 dataclass, with two big differences: Pydantic checks the types at runtime (a dataclass only documents them), and it converts where that is safe, such as "12" to 12 for an int field. Extra rules go in Field(...). If a Contentstack content type is the schema an editor's entry must follow, a Pydantic model is the same thing for a JSON body.

In [ ]:
# Pydantic models on their own: validation, conversion, errors, dumping.
from typing import Literal

from pydantic import BaseModel, Field, ValidationError


# A model is a class with typed fields, like a TypeScript interface that
# is also CHECKED at runtime. Think: a Contentstack content type schema.
class EntryCreate(BaseModel):
    title: str = Field(min_length=1, max_length=120)
    slug: str = Field(pattern=r"^[a-z0-9]+(-[a-z0-9]+)*$")
    content_type: Literal["blog_post", "product_page", "landing_page"]
    tags: list[str] = []  # safe here: Pydantic copies defaults per object
    views: int = 0


# Good data in: you get an object with attributes, not a dict.
e = EntryCreate(title="Navratri lookbook", slug="navratri-lookbook",
                content_type="blog_post", views="12")
print(e.views, type(e.views).__name__)
# -> 12 int   "12" was converted: lax mode accepts numeric strings

# model_dump() -> plain dict (for JSON or a DB row), like a spread {...e}
print(e.model_dump())
# -> {'title': 'Navratri lookbook', 'slug': 'navratri-lookbook',
#     'content_type': 'blog_post', 'tags': [], 'views': 12}

# Bad data in: ONE exception that lists EVERY problem, not just the first.
try:
    EntryCreate(title="", slug="Bad Slug", content_type="faq")
except ValidationError as exc:
    for err in exc.errors():
        print(err["loc"], err["type"])
# -> ('title',) string_too_short
# -> ('slug',) string_pattern_mismatch
# -> ('content_type',) literal_error

Three methods cover most of daily use. Model(**data) or Model.model_validate(data) builds and validates from a dict. obj.model_dump() turns it back into a dict, and obj.model_dump_json() into a JSON string. When validation fails you get one ValidationError listing every problem with its location (loc), a machine-readable type and a human msg. FastAPI sends exactly this list to the client as a 422.

FDE TIP: a list default such as tags: list[str] = [] is safe in a Pydantic model, because Pydantic copies the default for every new object. It is still the Day 2 mutable default trap in a plain function or a dataclass.

### Real-world FDE example

A retail customer's PIM system posts product updates to your service. One night a vendor feed starts sending price as "₹1,299" instead of 1299. Without a model, the bad string travels into Postgres and breaks the storefront sort order. With price: int in the model, those requests come back 422 with loc ['body', 'price'], your logs show exactly which feed is broken, and the good updates keep flowing.

### Common mistakes

- Expecting strict types by default. Pydantic's default (lax) mode turns "12" into 12 for an int field. If a field must reject strings, use Field(strict=True) or StrictInt.

- Using Pydantic v1 names from old tutorials: .dict(), .json(), parse_obj() and class Config. In Pydantic v2 they are model_dump(), model_dump_json(), model_validate() and model_config.

- One model for everything. A model that accepts uid and status from the client lets the client set them (Concept 4 splits them).

- Catching ValidationError and returning only the first error. The client then fixes one field per request.

## Concept 4: Request bodies, response models and status codes

### Why it matters for an FDE

This is where yesterday's design becomes behaviour. The design said POST returns 201 with a Location header, a duplicate slug is 409, an unknown uid is 404, and the client never sets the uid. FastAPI gives you one tool for each of those, and using them consistently is what makes the API feel predictable to the agency building against it.

### How it works

A function argument typed with a BaseModel is read from the JSON body. The function's return type is the response model: FastAPI validates what you return and keeps only that model's fields, so an internal column can never leak by accident. That is why real APIs use separate models: EntryCreate for what the client may send, EntryOut for what the server returns (including server-owned fields like uid and created_at). It is the same split as the props a React form submits versus the record the backend sends back.

For status codes, set the success code in the decorator (status_code=201) and raise HTTPException(status_code=404, detail=...) for errors. Raising stops the function at once, like throwing in JS; FastAPI catches it and sends {"detail": ...} with that code. To set a header such as Location, add a response: Response argument and write to response.headers.

In [ ]:
# Create and read with a body model, a response model and real status codes.
from datetime import UTC, datetime
from typing import Literal

from fastapi import FastAPI, HTTPException, Response, status
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field

ContentType = Literal["blog_post", "product_page", "landing_page"]


class EntryCreate(BaseModel):
    """What the CLIENT sends. No uid or timestamps: the server owns those."""
    title: str = Field(min_length=1, max_length=120)
    slug: str = Field(pattern=r"^[a-z0-9]+(-[a-z0-9]+)*$")
    content_type: ContentType


class EntryOut(EntryCreate):
    """What the SERVER returns: the input fields plus server-owned ones."""
    uid: str
    status: Literal["draft", "published"]
    created_at: datetime


app = FastAPI()
# An in-memory "database": uid -> dict. It resets on every restart.
# GOTCHA: fine for today; Day 24 swaps it for Postgres.
ENTRIES: dict[str, dict] = {}


@app.post("/v1/entries", status_code=status.HTTP_201_CREATED)
def create_entry(body: EntryCreate, response: Response) -> EntryOut:
    # A parameter typed with a BaseModel is read from the JSON body.
    if any(e["slug"] == body.slug for e in ENTRIES.values()):
        # raise, not return: FastAPI turns this into a 409 JSON response
        raise HTTPException(status_code=409,
                            detail=f"slug '{body.slug}' already exists")
    uid = f"blt{len(ENTRIES) + 1:03d}"  # blt001, blt002 ... (demo only)
    row = {**body.model_dump(), "uid": uid, "status": "draft",
           "created_at": datetime.now(UTC)}
    ENTRIES[uid] = row
    # Day 22: a 201 should say where the new resource lives.
    response.headers["Location"] = f"/v1/entries/{uid}"
    return EntryOut(**row)


# The return type doubles as the response model: FastAPI validates the
# output and FILTERS it to EntryOut's fields before sending.
@app.get("/v1/entries/{uid}")
def get_entry(uid: str) -> EntryOut:
    row = ENTRIES.get(uid)  # .get() returns None instead of KeyError
    if row is None:
        raise HTTPException(status_code=404, detail=f"entry {uid} not found")
    return EntryOut(**row)


client = TestClient(app)
new = {"title": "Diwali gift guide", "slug": "diwali-gift-guide",
       "content_type": "blog_post"}
r = client.post("/v1/entries", json=new)  # json= sets Content-Type for us
print(r.status_code, r.headers["location"])
# -> 201 /v1/entries/blt001
print(r.json()["status"], r.json()["created_at"][:4])
# -> draft 2026   datetime became an ISO 8601 string in the JSON
print(client.post("/v1/entries", json=new).json())
# -> {'detail': "slug 'diwali-gift-guide' already exists"}  (status 409)
print(client.get("/v1/entries/blt999").status_code)
# -> 404

### Real-world FDE example

A bank's content team uses your API to manage FAQ entries, and their security review asks one question first: "can a caller make something live without approval?" Because EntryCreate has no status field and the server always sets "draft", the answer is no by construction, and you can show it in /docs in ten seconds. Publishing is a separate POST /publish action that Day 26 will put behind a stronger API key.

### Common mistakes

- Returning {"error": "not found"} with a 200 instead of raising HTTPException. Every client counts it as a success (Day 22).

- Writing return HTTPException(...) instead of raise. FastAPI then tries to send the exception object as a 200 body.

- Forgetting status_code=201 on create routes. FastAPI's default is 200 for every route.

- Letting the client send server-owned fields (uid, status, version, created_at) because the same model is used for input and output.

- Checking the duplicate slug after generating the uid, so failed requests burn ids.

## Concept 5: Updates, deletes and validation errors

### Why it matters for an FDE

Partial updates are where most home-made APIs lose data. A PATCH that changes the title must not wipe the tags, and a client that sends nothing must change nothing. The 422 error body is the other half: it is what the customer's developers will see most often while integrating, so you need to read it at a glance and explain it on a call.

### How it works

For PATCH, make a separate model where every field is optional with default None. Then body.model_dump(exclude_unset=True) returns only the keys the client actually sent, and you merge those into the stored row. Note the difference: a field the client did not send is unset; a field sent as null is set to None. exclude_unset keeps the second kind, so decide what null means (today's task rejects it with a 422).

For DELETE, use status_code=204 and return nothing: 204 means the body is empty. A FastAPI validation error always has the same shape: status 422 and {"detail": [ ... ]}, one item per problem, each with type, loc, msg and input. The first element of loc says where the bad value was: "body", "query", "path" or "header"; the rest is the field path.

In [ ]:
# PATCH with exclude_unset, DELETE with 204, and the 422 error shape.
from fastapi import FastAPI, HTTPException, status
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field


class EntryUpdate(BaseModel):
    """PATCH body: every field optional, because the client sends only
    what changes (Day 22: PATCH = change only the fields I send)."""
    title: str | None = Field(default=None, min_length=1, max_length=120)
    tags: list[str] | None = None


class EntryOut(BaseModel):
    uid: str
    title: str
    tags: list[str]


app = FastAPI()
ENTRIES = {"blt001": {"uid": "blt001", "title": "Old title",
                      "tags": ["sale"], "internal_note": "do not leak"}}


@app.patch("/v1/entries/{uid}")
def update_entry(uid: str, body: EntryUpdate) -> EntryOut:
    if uid not in ENTRIES:
        raise HTTPException(status_code=404, detail=f"entry {uid} not found")
    # exclude_unset=True keeps ONLY the keys the client actually sent.
    changes = body.model_dump(exclude_unset=True)
    print("changes:", changes)
    ENTRIES[uid].update(changes)  # dict.update merges, like {...a, ...b}
    # The row still holds internal_note; returning it as EntryOut drops it.
    return EntryOut(**ENTRIES[uid])


# 204 No Content: success with an empty body (Day 22).
@app.delete("/v1/entries/{uid}", status_code=status.HTTP_204_NO_CONTENT)
def delete_entry(uid: str) -> None:
    if ENTRIES.pop(uid, None) is None:  # pop with a default never raises
        raise HTTPException(status_code=404, detail=f"entry {uid} not found")


client = TestClient(app)
r = client.patch("/v1/entries/blt001", json={"title": "New title"})
# -> changes: {'title': 'New title'}   tags was not sent, so not touched
print(r.json())
# -> {'uid': 'blt001', 'title': 'New title', 'tags': ['sale']}
#    no internal_note: the response model filtered it out

bad = client.patch("/v1/entries/blt001", json={"title": ""})
print(bad.status_code)  # -> 422
err = bad.json()["detail"][0]  # FastAPI's validation error shape
print(err["loc"], err["msg"])
# -> ['body', 'title'] String should have at least 1 character

r = client.delete("/v1/entries/blt001")
print(r.status_code, repr(r.text))  # -> 204 ''
print(client.delete("/v1/entries/blt001").status_code)
# -> 404   DELETE is idempotent: the entry stays gone, the code differs

GOTCHA: FastAPI's own errors use two shapes: {"detail": "text"} for your HTTPExceptions and {"detail": [list]} for validation. Yesterday's design chose one problem-details shape for everything; Day 25 adds the exception handlers that make FastAPI follow it.

### Real-world FDE example

A publishing customer's editors used a dashboard that sent the whole entry on every save. When two editors worked on one article, the second save restored the first editor's old tags. Switching the dashboard to PATCH with only the changed fields, and the API to exclude_unset, fixed the lost edits without any locking. Add the version field from today's task and Day 22's If-Match, and the remaining conflicts become a clear 412.

### Common mistakes

- Using model_dump() without exclude_unset in a PATCH, which writes None over every field the client did not send.

- Reusing EntryCreate for PATCH, which makes every field required and turns a one-field change into a 422.

- Returning a body with a 204. Clients and proxies are allowed to ignore it, and some break.

- Reading only msg from a 422. loc tells you which field and where it came from; read it first.

## Check your understanding (predict first, then run)

Save the file below as day23/scratch/predict.py. Before running it, write in your notebook what each of the six lines will print, and why. Then run it with python3 predict.py and explain any surprise in one sentence.

In [ ]:
# Six predictions. Write each answer down BEFORE you run the file.
from typing import Annotated

from fastapi import FastAPI, HTTPException, Query
from fastapi.testclient import TestClient
from pydantic import BaseModel


class Entry(BaseModel):
    title: str
    views: int = 0


app = FastAPI()


@app.get("/v1/entries/{uid}")
def get_entry(uid: str) -> dict:
    raise HTTPException(status_code=404, detail=f"entry {uid} not found")


@app.get("/v1/entries/latest")  # declared AFTER the {uid} route
def latest() -> dict:
    return {"uid": "blt042"}


@app.get("/v1/count")
def count(limit: Annotated[int, Query(ge=1)] = 10) -> dict:
    return {"limit": limit}


@app.post("/v1/entries")
def create(entry: Entry) -> Entry:
    return entry


c = TestClient(app)
print("Q1", c.get("/v1/entries/latest").json())
print("Q2", c.get("/v1/count?limit=0").status_code)
print("Q3", c.get("/v1/count?limit=5").json())
r4 = c.post("/v1/entries", json={"views": 3})
print("Q4", r4.json()["detail"][0]["loc"])
print("Q5", c.post("/v1/entries", json={"title": "Hi", "views": "7"}).json())
print("Q6", c.post("/v1/entries", content="title=Hi").status_code)

## Hands-on task (about 2 hours): day23/content_api.py and smoke.sh

Scenario: Acme Retail's partner agency is ready to build against the design you wrote yesterday. Your job today is the entries part of it as a running service. Data lives in a Python dict for now; tomorrow the same URLs are backed by Postgres.

Part A (90 min): open day23/content_api.py. The models EntryOut and EntryPage, the store, reset_store(), new_uid() and /health are given. You write the two input models (EntryCreate, EntryUpdate) and seven functions: find_or_404, create_entry, list_entries, get_entry, update_entry, delete_entry and publish_entry. Each has a comment block with the exact rules and an example, plus step-by-step hints. Work one route at a time and rerun the tests after each: python3 -m pytest -q -k create runs only the tests whose name contains "create".

| Method and path | Success | Errors |
|---|---|---|
| POST /v1/entries | 201 + Location | 409 duplicate slug, 422 |
| GET /v1/entries | 200 {items, next_cursor} | 422 bad limit or status |
| GET /v1/entries/{uid} | 200 | 404 |
| PATCH /v1/entries/{uid} | 200, version + 1 | 404, 422 (also for null) |
| DELETE /v1/entries/{uid} | 204, empty body | 404 |
| POST /v1/entries/{uid}/publish | 200, version + 1 | 404, 409 already published |

Part B (30 min): start the server with fastapi dev content_api.py, open /docs and run every route once with "Try it out". Then fill in day23/smoke.sh: nine steps, each with the output it must print; write one curl command under each TODO, using Day 22's flags.

Acceptance criteria:

- python3 -m pytest -q inside day23/ shows 26 passed. Do not edit test_content_api.py.

- /docs lists all seven routes with their models, and you have run each one once with "Try it out".

- bash smoke.sh, against a freshly started dev server, prints every step's expected output from top to bottom.

- No input you can think of produces a 500: bad input is 422, an unknown uid is 404, a clash with the current state is 409.

- EntryCreate has no uid, status, version or timestamp fields: the server owns them.

- ruff check . and mypy content_api.py are clean (Day 12).

- Committed and pushed with a message such as "Day 23: in-memory content API with FastAPI" (a WIP push counts).

Hints, not answers:

- Write the models first and run the tests: the six test_create_invalid_field_is_422 cases should pass before you write any route, because validation happens before your function runs.

- When a test fails, read the assertion line and the response FastAPI printed. Add print(r.json()) in the test temporarily if you need the full body, and remove it afterwards.

- If a route returns 500 in the tests, the traceback in the pytest output points at the exact line; a common cause is returning a row dict that is missing a field EntryOut needs.

- For list_entries, build the filtered list first, then slice it. next_cursor is the uid of the last item on the page, but only when more items exist after it.

- For update_entry, the null check must run before you change the row; otherwise the 422 leaves the entry half updated.

Stretch (only if Part A, Part B and the catch-up item are pushed): add GET /v1/entries/by-slug/{slug} and make sure it is declared so that it cannot be swallowed by /v1/entries/{uid}. Add two tests of your own for it in a new file test_mine.py.

Catch-up (1h): if day22/api_design.md is not pushed, finish it first: today's task is easier when the design is written down. Otherwise continue with Day 15's schema.sql and queries.sql, which tomorrow's Postgres work depends on.

## DSA practice (30 min)

Both problems are in day23/dsa.py with examples and hints. Give each 15 minutes before reading the hints.

- match_route: given a pattern such as /v1/entries/{uid}/tags/{tag} and a path such as /v1/entries/blt7/tags/sale, return {'uid': 'blt7', 'tag': 'sale'}, or None if the path does not match. This is a small version of what FastAPI does for every request. Hint: split both on "/" and compare segment counts first.

- unique_slug: turn "Diwali Gift Guide" into diwali-gift-guide, and if that is already taken return diwali-gift-guide-2, then -3, and so on. Practises sets and while loops, and it is the friendly alternative to today's 409. Hint: a regular expression can find the runs of letters and digits for you.

## Quiz (3 quick questions)

- Why does the task use separate EntryCreate, EntryUpdate and EntryOut models instead of one Entry model? Give two reasons.

- GET /v1/entries/{uid} is declared before GET /v1/entries/latest. What does a request to /v1/entries/latest return, and what are two ways to fix it?

- In a PATCH handler, what is the difference between body.model_dump() and body.model_dump(exclude_unset=True) for the body {"title": "New"}, and why does it matter?

## What you achieved today

You built your first real web service: routes from decorators, typed path and query parameters, Pydantic models for input and output, the status codes from yesterday's design, partial updates that cannot lose data, and interactive documentation you did not have to write. The content API is the core of the Phase 1 milestone, and it is the same skeleton every AI service in the next phases will use.

Tomorrow, Day 24: FastAPI with Postgres. The same routes, backed by your Week 3 schema through a connection pool, with settings from environment variables and a repository layer, so the data survives a restart.

## When you're done

Paste into the Forward Deployment Engineer project chat as Day 23 review: content_api.py, the last lines of your pytest output, the smoke.sh output, your six predictions with the real results, your quiz answers, dsa.py, and the link to your latest commit. Say how far the catch-up item got.

## Recap answers

*Your answers:*
